In [1]:
import pandas as pd
import glob
import os

In [2]:
path_aiming = 'C:/CourseWork/Dissertation Classifying grip strategies using machine learning/data/01_raw/Aiming/filtered_data'
path_prehension = 'C:/CourseWork/Dissertation Classifying grip strategies using machine learning/data/01_raw/Prehension/filtered_data'
path_visual_illusion = 'C:/CourseWork/Dissertation Classifying grip strategies using machine learning/data/01_raw/Visual Illusions/filtered_data'

In [ ]:
def create_perfect_master_dataset(base_paths):
    """
    Loads, merges, cleans, and unifies all data sources into a single,
    model-ready Parquet file. This version automatically detects and categorizes
    all .csv files.
    """

    # --- Part 1: Dynamically Load All Parameter and Trajectory Files ---
    print("--- Part 1: Finding and categorizing all .csv files ---")
    
    all_param_files = []
    all_traj_files = []
    
    for dataset_name, base_path in base_paths.items():
        all_csv_files = glob.glob(os.path.join(base_path, '*.csv'))
        print(f"Found {len(all_csv_files)} total .csv files in '{dataset_name}'")
        
        for f in all_csv_files:
            if 'trajData.csv' in os.path.basename(f):
                all_traj_files.append((dataset_name, f))
            else:
                all_param_files.append(f)
                
    print(f"\nIdentified {len(all_param_files)} parameter/static files to merge.")
    print(f"Identified {len(all_traj_files)} trajectory files to process.")

    all_param_dfs = []
    for file_path in all_param_files:
        try:
            df = pd.read_csv(file_path, low_memory=False)
            source_name = os.path.basename(file_path).replace('.csv', '')
            df['param_source_file'] = source_name 
            all_param_dfs.append(df)
        except Exception as e:
            print(f"Could not load or process {file_path}. Error: {e}")
            
    # Combine all parameter and info dataframes together
    param_df = pd.concat(all_param_dfs, ignore_index=True)

    # Consolidate by trial. This is critical for merging data from different
    # files (e.g., grasp_paramData, reach_paramData, AimingData) for the same trial.
    # It assumes 'subjName' and 'trialN' exist in all parameter-like files.
    print("\nConsolidating all parameter data from multiple sources...")
    if 'subjName' in param_df.columns and 'trialN' in param_df.columns:
        param_df = param_df.groupby(['subjName', 'trialN']).first().reset_index()
        print(f"Loaded and consolidated {len(param_df)} unique trial parameter rows.")
    else:
        print("Warning: 'subjName' or 'trialN' not found in all parameter files. Skipping consolidation.")


    # --- Part 2: Process Trajectory Data Iteratively and Merge ---
    print("\n--- Part 2: Loading and merging Trajectory (trajData) files one by one ---")
    all_merged_dfs = []
    merge_keys = ['subjName', 'trialN']
    
    print(f"Processing {len(all_traj_files)} trajData files...")
    for dataset_name, file_path in all_traj_files:
        traj_df = pd.read_csv(file_path, low_memory=False)
        traj_df['dataset_source'] = dataset_name
        
        # Merge with the consolidated parameter dataframe
        current_subject = traj_df['subjName'].iloc[0]
        subject_param_df = param_df[param_df['subjName'] == current_subject]
        
        # Avoid duplicate columns after merge
        cols_to_drop_from_params = [col for col in traj_df.columns if col in subject_param_df.columns and col not in merge_keys]
        subject_param_df_unique = subject_param_df.drop(columns=cols_to_drop_from_params, errors='ignore')
        
        merged_df = pd.merge(traj_df, subject_param_df_unique, on=merge_keys, how='left')
        all_merged_dfs.append(merged_df)

    print("\n--- Part 3: Concatenating all processed pieces ---")
    master_df = pd.concat(all_merged_dfs, ignore_index=True)
    print(f"Total rows in dataframe: {len(master_df)}")

    print("\n--- Part 4: Unifying sequential features ---")
    markers = ['index', 'thumb', 'wrist']
    axes = ['X', 'Y', 'Z']
    unified_seq_features = []
    for marker in markers:
        for axis in axes:
            raw_col = f'{marker}{axis}raw'
            proc_col = f'{marker}{axis}'
            unified_col = f'{marker}{axis}_unified'
            master_df[unified_col] = master_df[raw_col].fillna(master_df[proc_col])
            unified_seq_features.append(unified_col)
    null_check = master_df[unified_seq_features].isnull().sum()
    print("Nulls remaining in unified sequential features:\n", null_check)

    print("\n--- Part 5: Cleaning and selecting static features ---")
    categorical_static = ['signal'] 
    master_df = pd.get_dummies(master_df, columns=categorical_static, prefix=categorical_static, dummy_na=True)
    numeric_static = ['FX', 'FY', 'FZ', 'FVel', 'FAcc', 'MVel', 'MAcc', 'MDec', 'pathLength', 'MGA', 'timeMGA', 'movTime']
    encoded_cols = [col for col in master_df.columns if any(cat in col for cat in categorical_static)]
    final_static_features = numeric_static + encoded_cols
    
    # Ensure all selected static feature columns exist before filling NA
    existing_static_features = [col for col in final_static_features if col in master_df.columns]
    missing_static_features = set(final_static_features) - set(existing_static_features)
    if missing_static_features:
        print(f"Warning: The following expected static columns were not found and will be ignored: {missing_static_features}")
        
    master_df[existing_static_features] = master_df[existing_static_features].fillna(0)
    print(f"Selected {len(existing_static_features)} static features.")

    print("\n--- Part 6: Engineering and cleaning labels ---")
    label_config = {
        'aiming': ['visCond', 'surface', 'distance'],
        'prehension': ['visCond', 'surface', 'distance'],
        'visual_illusion': ['visCond', 'illusion', 'targetPos', 'targetSize']
    }
    master_df['grip_strategy_label'] = ''
    for dataset_name, cols in label_config.items():
        if all(c in master_df.columns for c in cols):
            mask = master_df['dataset_source'] == dataset_name
            if mask.sum() > 0:
                conditions_str = master_df.loc[mask, cols].fillna('NA').astype(str).agg('_'.join, axis=1)
                master_df.loc[mask, 'grip_strategy_label'] = f"{dataset_name}_" + conditions_str
    
    master_df['grip_strategy_label'] = master_df['grip_strategy_label'].str.replace('.csv', '', regex=False)
    master_df['grip_strategy_label'] = master_df['grip_strategy_label'].str.replace('_woord_', '_wood_', regex=False)
    print(f"Final number of unique classes: {master_df['grip_strategy_label'].nunique()}")

    #  Part 7: Selecting final columns and saving to Parquet 
    final_identifiers = ['subjName', 'trialN']
    all_possible_columns = final_identifiers + unified_seq_features + existing_static_features + ['grip_strategy_label']
    final_columns = [col for col in all_possible_columns if col in master_df.columns]

    # Create an explicit copy to avoid the warning
    final_df = master_df[final_columns].copy() 

    # Now, drop rows with nulls from the new DataFrame
    final_df.dropna(subset=unified_seq_features, inplace=True)

    print(f"Final dataframe shape after dropping any remaining nulls: {final_df.shape}")

    parquet_filename = "comprehensive_master_data_universal.parquet"
    final_df.to_parquet(parquet_filename, engine='pyarrow', compression='snappy')
    print(f"\nDone! The perfect '{parquet_filename}' has been created.")
    print("It contains only the necessary columns for modeling.")

In [4]:
if __name__ == '__main__':
    try:
        import pyarrow
    except ImportError:
        print("Error: 'pyarrow' library not found. Please install it: pip install pyarrow")
    else:
        path_aiming = 'C:/CourseWork/Dissertation Classifying grip strategies using machine learning/data/01_raw/Aiming/filtered_data'
        path_prehension = 'C:/CourseWork/Dissertation Classifying grip strategies using machine learning/data/01_raw/Prehension/filtered_data'
        path_visual_illusion = 'C:/CourseWork/Dissertation Classifying grip strategies using machine learning/data/01_raw/Visual Illusions/filtered_data'

        paths_to_process = {
            'aiming': path_aiming,
            'prehension': path_prehension,
            'visual_illusion': path_visual_illusion
        }

        all_paths_exist = True
        for name, path in paths_to_process.items():
            if not os.path.isdir(path):
                print(f"FATAL ERROR: The path for '{name}' does not exist or is not a directory. Path checked: {path}")
                all_paths_exist = False

        if all_paths_exist:
            print("All paths found. Starting dataset creation...")
            create_perfect_master_dataset(paths_to_process)
        else:
            print("\nAborting due to missing paths. Please correct the paths in the script.")


All paths found. Starting dataset creation...
--- Part 1: Finding and categorizing all .csv files ---
Found 73 total .csv files in 'aiming'
Found 81 total .csv files in 'prehension'
Found 80 total .csv files in 'visual_illusion'

Identified 176 parameter/static files to merge.
Identified 58 trajectory files to process.

Consolidating all parameter data from multiple sources...
Loaded and consolidated 2876 unique trial parameter rows.

--- Part 2: Loading and merging Trajectory (trajData) files one by one ---
Processing 58 trajData files...

--- Part 3: Concatenating all processed pieces ---
Total rows in dataframe: 2836573

--- Part 4: Unifying sequential features ---
Nulls remaining in unified sequential features:
 indexX_unified    0
indexY_unified    0
indexZ_unified    0
thumbX_unified    0
thumbY_unified    0
thumbZ_unified    0
wristX_unified    0
wristY_unified    0
wristZ_unified    0
dtype: int64

--- Part 5: Cleaning and selecting static features ---
Selected 14 static featur